# TASK 7.7 — Xây dữ liệu v2 và kiểm toán trước bàn giao

Hợp đồng dữ liệu mục tiêu: 14.400 ảnh (5.350 fake mới, 6.050 real mới, 3.000 replay in-dist).
Mọi lệnh dùng subprocess.run(check=True). Thiếu reference, replay bị deny-list chặn,
constraint sai hoặc audit FAIL đều dừng quy trình. Không mở Task 7.11 khi còn lỗi.

**Chính sách Replay v2 đã chốt**: Đã loại bỏ 600 ảnh Midjourney khỏi Replay ở cấp bản ghi để bảo đảm 100% generator-disjoint với CFE test. Kho Replay giữ đúng 3.000 mẫu sạch (1.800 Real ImageNet + 1.200 SD 1.5, chiếm 20,83% tổng kho 14.400).

---  
## BƯỚC 1: Mount Google Drive & Thiết Lập Thư Mục Làm Việc

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, subprocess, json, shutil

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Cau hinh lay truc tiep ma nguon tu nhanh hoang
REPO_URL = 'https://github.com/LeeVHoangtk3/fatformer-xla.git'
BRANCH = 'hoang'
PROJECT_ROOT = Path('/content/drive/MyDrive/Fatformer/fatformer-xla')

# 3. Tu dong Clone neu chua co, hoac Pull neu da ton tai
PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
if not (PROJECT_ROOT / '.git').exists():
    if PROJECT_ROOT.exists():
        print(f'[*] Thu muc {PROJECT_ROOT} da ton tai nhung thieu .git. Dang don dep de clone moi...')
        shutil.rmtree(PROJECT_ROOT)
    print(f'[*] Dang lay ma nguon truc tiep tu nhanh {BRANCH} ({REPO_URL})...')
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    print(f'[*] Dang keo cap nhat moi nhat truc tiep tu nhanh {BRANCH}...')
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=PROJECT_ROOT, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=PROJECT_ROOT, check=True)
    subprocess.run(['git', 'pull', 'origin', BRANCH], cwd=PROJECT_ROOT, check=True)

# 4. Xac nhan tep cot loi build_manifest_v2.py
if not (PROJECT_ROOT / 'tools/build_manifest_v2.py').is_file():
    raise RuntimeError(f'Loi: Khong tim thay tools/build_manifest_v2.py tai {PROJECT_ROOT} sau khi keo nhanh {BRANCH}!')

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'tools'))

def run(*args):
    subprocess.run([str(x) for x in args], check=True, cwd=PROJECT_ROOT)

print(f'[✓] Da dong bo thanh cong workspace tai: {PROJECT_ROOT}')
run('git', 'log', '-1', '--oneline')
BASE_CONFIG = 'configs/phase7_data_v2.yaml'
CONFIG = BASE_CONFIG

---  
## BƯỚC 2: Cài Đặt Thư Viện Phụ Trợ & Kiểm Tra Môi Trường

In [ ]:
run(sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'duckdb', 'pyarrow', 'pillow', 'scipy', 'pyyaml', 'requests', 'matplotlib')
import duckdb, pyarrow, PIL, scipy, numpy, yaml
print({m.__name__: m.__version__ for m in (duckdb, pyarrow, PIL, scipy, numpy)})
# Fingerprint của config/deny-list/replay thay đổi sẽ chặn resume.

## Bước 3 — Kiểm thử offline
Phải thành công trước khi chạy dữ liệu thật. Đây là kiểm thử cơ chế trên dữ liệu tổng hợp, không thay thế audit thực tế.

In [ ]:
run(sys.executable, 'tools/test_data_v2.py', '-v')

---  
## BƯỚC 4: Tạo Reference Hash Index Của Các Tập Test & Sinh Deny-List
> **Mục đích**: Tạo cơ sở dữ liệu băm (SHA-256 + pHash 64-bit) đọc trực tiếp từ các file `.tar` trên Google Drive nhằm phục vụ bộ lọc kiểm toán cách ly.

In [ ]:
import tarfile
DRIVE_DATASETS = Path('/content/drive/MyDrive/Fatformer/datasets')
TAR_CFE_TEST = DRIVE_DATASETS / 'test_communityforensics.tar'
TAR_GANS_TEST = DRIVE_DATASETS / 'test_benchmark_gans.tar'
TAR_DIFF_TEST = DRIVE_DATASETS / 'test_benchmark_diffusion.tar'
TAR_STAGING = DRIVE_DATASETS / 'diffusion_staging.tar'
for path in (TAR_CFE_TEST, TAR_GANS_TEST, TAR_DIFF_TEST, TAR_STAGING):
    if not path.is_file():
        raise FileNotFoundError(path)

# Đọc đúng một manifest từ tar, không bỏ qua lỗi thiếu metadata.
REF_DIR = PROJECT_ROOT / 'DATASET/logs/phase7'
REF_DIR.mkdir(parents=True, exist_ok=True)
CFE_MANIFEST = REF_DIR / 'cfe_test_manifest.json'
with tarfile.open(TAR_CFE_TEST, 'r:*') as archive:
    candidates = [m for m in archive if m.isfile() and Path(m.name).name == 'cfe_manifest.json']
    if len(candidates) != 1:
        raise RuntimeError(f'Cần đúng một cfe_manifest.json; tìm thấy {len(candidates)}')
    with archive.extractfile(candidates[0]) as source:
        CFE_MANIFEST.write_bytes(source.read())

# Sinh deny-list và kiểm tra replay trước khi hash/tải dữ liệu lớn.
run(sys.executable, 'tools/build_generator_denylist.py',
    '--cfe-manifest', CFE_MANIFEST,
    '--task51-report', 'docs/bao_cao/C/bao_cao_task_5.1_full_benchmark_18_datasets.md',
    '--out', REF_DIR / 'generator_denylist.json')
from build_manifest_v2 import Config, check_replay_generators
check_replay_generators(Config(str(PROJECT_ROOT / CONFIG)))

run(sys.executable, 'tools/index_test_sets.py', '--role', 'cfe',
    '--tar', TAR_CFE_TEST, '--manifest', CFE_MANIFEST, '--workers', '4',
    '--out', REF_DIR / 'hashes_cfe_test.json')
run(sys.executable, 'tools/index_test_sets.py', '--role', 'test18',
    '--tar', TAR_GANS_TEST, TAR_DIFF_TEST, '--workers', '4',
    '--out', REF_DIR / 'hashes_18sets.json.gz')

## Bước 5 — Preflight và dry-run
Builder kiểm tra replay với deny-list trước khi đọc metadata hoặc tải shard.
Nếu Midjourney replay bị chặn, dừng để giải quyết hợp đồng dữ liệu; không dùng --force để bỏ qua.

Số shard, dung lượng và thời gian phải lấy từ log thực tế. Không coi 14 shard/21,81 GB là cam kết cố định.
State cũ thiếu fingerprint phải chuyển sang thư mục state mới hoặc bắt đầu lại có chủ đích bằng --force.

In [ ]:
# Pin revision trước khi đọc metadata. Giữ nguyên config gốc trong Git.
from build_manifest_v2 import Config, check_replay_generators
check_replay_generators(Config(str(PROJECT_ROOT / CONFIG)))
import requests, re
RUNTIME_CONFIG = PROJECT_ROOT / 'configs/phase7_data_v2.runtime.yaml'
if RUNTIME_CONFIG.exists():
    runtime = yaml.safe_load(RUNTIME_CONFIG.read_text(encoding='utf-8'))
    base = yaml.safe_load((PROJECT_ROOT / BASE_CONFIG).read_text(encoding='utf-8'))
    check = json.loads(json.dumps(runtime))
    for name, spec in check['datasets'].items():
        spec['url_template'] = base['datasets'][name]['url_template']
    if check != base:
        raise RuntimeError('Config nguồn đã đổi. Tạo runtime/state mới; không tiếp tục resume cũ.')
else:
    runtime = yaml.safe_load((PROJECT_ROOT / BASE_CONFIG).read_text(encoding='utf-8'))
    for spec in runtime['datasets'].values():
        url = spec['url_template']
        match = re.match(r'https://huggingface.co/datasets/([^/]+/[^/]+)/resolve/main/', url)
        if match:
            response = requests.get(f'https://huggingface.co/api/datasets/{match.group(1)}', timeout=60)
            response.raise_for_status()
            revision = response.json()['sha']
            if not re.fullmatch(r'[0-9a-f]{40}', revision):
                raise RuntimeError('Dataset revision không hợp lệ')
            spec['url_template'] = url.replace('/resolve/main/', f'/resolve/{revision}/')
    RUNTIME_CONFIG.write_text(yaml.safe_dump(runtime, allow_unicode=True, sort_keys=False), encoding='utf-8')
CONFIG = str(RUNTIME_CONFIG)
run(sys.executable, 'tools/build_manifest_v2.py', '--config', CONFIG, '--dry-run')

---  
## BƯỚC 6: Chuẩn Bị Dữ Liệu Replay In-dist (3.600 Mẫu)
> **Mục đích**: Giải nén tập `diffusion_staging.tar` lưu trên Drive vào bộ nhớ cục bộ `/content/dataset_local/diffusion_staging`.

In [ ]:
REPLAY_DIR = Path('/content/dataset_local/diffusion_staging')
REPLAY_DIR.mkdir(parents=True, exist_ok=True)
run('tar', '-xf', TAR_STAGING, '-C', REPLAY_DIR)
print('Replay đã giải nén; builder sẽ xác minh đủ ảnh, checksum và generator.')

## Bước 7 — Build với đầy đủ reference
Ảnh nguyên byte và state nằm trong repo trên Drive; cache Parquet tạm nằm ở /content.
Resume xác minh ảnh và fingerprint trước khi bỏ qua shard. Giới hạn 4 giờ/60 GB được kiểm tra lúc tải/xử lý;
khi chạm ngưỡng, dừng và lưu checkpoint, không tự hạ 8.000 ảnh rồi công nhận đạt.

Dữ liệu mới được lọc với cả CFE và 18 tập test trước chia split. Split thử chia theo generator;
ngoại lệ theo ảnh được ghi trong metadata. Có manifest train_select để loader hiện tại không trộn val_calib.

In [ ]:
run(sys.executable, 'tools/build_manifest_v2.py', '--config', CONFIG,
    '--cache_dir', '/content/cache_v2', '--replay_root', REPLAY_DIR,
    '--replay_copy_mode', 'copy',
    '--ref_index', REF_DIR / 'hashes_cfe_test.json', REF_DIR / 'hashes_18sets.json.gz',
    '--resume', '--strict')

---  
## BƯỚC 8: Kiểm Toán Độc Lập Chốt Chặn G7.2 (`tools/audit_manifest_v2.py`)
> **Ràng buộc bất di bất dịch**: Báo cáo kiểm toán bắt buộc phải có **`violations = 0`** và **`failures = 0`**. Nếu có bất kỳ vi phạm nào, Task 7.11 sẽ bị chặn ngay lập tức.

In [ ]:
run(sys.executable, 'tools/audit_manifest_v2.py', '--config', CONFIG,
    '--manifest', 'DATASET/manifests/data_manifest_v2.json',
    '--ref', f'cfe={REF_DIR / "hashes_cfe_test.json"}',
    '--ref', f'test18={REF_DIR / "hashes_18sets.json.gz"}',
    '--threshold_external', '8', '--threshold_internal', '3',
    '--out', REF_DIR / 'data_isolation_report.json')

from package_manifest_v2 import validate_handoff
validate_handoff('DATASET/manifests/data_manifest_v2.json',
                 REF_DIR / 'data_isolation_report.json', CONFIG)
print('Audit đầy đủ và khớp manifest/config hiện tại.')

---  
## BƯỚC 9: Đóng Gói `train_crossdomain_v2.tar` & Đẩy Lên Google Drive
> **Mục đích**: Đóng gói toàn bộ byte ảnh gốc vào kho `.tar` không nén lại (để đọc tuần tự tốc độ cao) và lưu trực tiếp lên Google Drive kèm chữ ký băm SHA-256.

In [ ]:
DEST_DRIVE_PHASE7 = Path('/content/drive/MyDrive/Fatformer/phase7')
TAR_OUT = DEST_DRIVE_PHASE7 / 'train_crossdomain_v2.tar'
# Chỉ đóng gói record đã chọn và có báo cáo audit khớp checksum.
# Tar giữ đường dẫn DATASET/... để extract -C <repo>.
run(sys.executable, 'tools/package_manifest_v2.py', '--config', CONFIG,
    '--manifest', 'DATASET/manifests/data_manifest_v2.json',
    '--report', REF_DIR / 'data_isolation_report.json', '--out', TAR_OUT)
print(f'Đã tạo {TAR_OUT} và checksum; vẫn cần C xác nhận tiếp nhận.')

## Bước 10 — Bàn giao có bằng chứng

Chỉ hoàn tất khi tar và checksum tồn tại, audit khớp manifest, báo cáo thực tế đã viết và C xác nhận.
Không đánh dấu hoàn thành bằng checklist cố định.

- Extract: `tar -xf train_crossdomain_v2.tar -C <repo>`.
- Train/chọn checkpoint với loader hiện tại: truyền `--train_manifest DATASET/manifests/data_manifest_v2_train_select.json`.
  File này chỉ chứa train và val_select; không đưa val_calib vào loader đang gộp mọi split khác train thành val.
- Calibration/router chỉ đọc `data_manifest_v2_val_calib.json`.
- Lưu policy chia split, số lượng sau mọi bộ lọc, thời gian/dung lượng thật và kết quả audit trong báo cáo Task 7.7.
- Nếu còn xung đột replay/generator hoặc thiếu reference: Task 7.7 và 7.11 vẫn bị chặn.